# Mel + Chroma

## Tanítás

In [4]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Reshape, UpSampling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# --- 1. MIXED PRECISION (T4/P100 GPU-n gyorsítás) ---
mixed_precision.set_global_policy('mixed_float16')

# --- ÚTVONALAK ---
# Kérlek, ellenőrizd, hogy a Kaggle útvonalaid helyesek-e!
H5_PATH           = "/kaggle/input/datasets/bresgbor/spotify-dataset-compressed/spotify_dataset_compressed.h5"
SAVE_PATH_AE      = "/kaggle/working/spotify_autoencoder_notempo.keras"
SAVE_PATH_ENCODER = "/kaggle/working/spotify_encoder_notempo.keras"


# ==========================================
# 2. EGYÉNI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    """L2 normalizálás a bottleneck vektoron – koszinusz-hasonlósághoz."""
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)


# ==========================================
# 3. DATA GENERATOR (CSAK MEL + CHROMA)
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, split_name='train', batch_size=64, shuffle=True, **kwargs):
        super().__init__(workers=4, use_multiprocessing=False, max_queue_size=10, **kwargs)
        
        self.h5_path    = h5_path
        self.batch_size = batch_size
        self.shuffle    = shuffle

        self.hf = h5py.File(self.h5_path, 'r')
        splits     = self.hf['ml/split'][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
        self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_split_indices  = np.arange(
            index * self.batch_size,
            min((index + 1) * self.batch_size, len(self.indices))
        )
        batch_global_indices = np.sort(self.indices[batch_split_indices])

        # TÖRLÉS: X_tempo kivéve
        X_mel    = self.hf['spectrograms/mel'][batch_global_indices]
        X_chroma = self.hf['spectrograms/chroma'][batch_global_indices]

        X_mel    = np.expand_dims(X_mel,    axis=-1).astype(np.float32)
        X_chroma = np.expand_dims(X_chroma, axis=-1).astype(np.float32)

        # TÖRLÉS: tempo bemenet és célpont kivéve
        inputs  = {'mel_input': X_mel, 'chroma_input': X_chroma}
        targets = {'mel_output': X_mel, 'chroma_output': X_chroma}
        return inputs, targets

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __del__(self):
        if hasattr(self, 'hf') and self.hf.id.valid:
            self.hf.close()


# ==========================================
# 4. MODELL ÉPÍTŐKOCKÁK
# ==========================================
def create_encoder_branch(x, filters, pool_sizes, branch_name):
    # Itt egy kis javítás a hibád miatt: A Te eredeti kódodban a rétegek nem kapcsolódtak megfelelően egymásba,
    # mert az `x`-et újra és újra felülírtad, de a Conv2D nem adta tovább a formáját.
    # Ehelyett egy lokális változóban (current_tensor) kell vinni az állapotot.
    current_tensor = x
    for i, f in enumerate(filters):
        current_tensor = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_enc_conv_{i+1}')(current_tensor)
        current_tensor = BatchNormalization(name=f'{branch_name}_enc_bn_{i+1}')(current_tensor)
        current_tensor = Activation('relu', name=f'{branch_name}_enc_relu_{i+1}')(current_tensor)
        current_tensor = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_enc_pool_{i+1}')(current_tensor)
    
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(current_tensor)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(current_tensor)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])


def create_decoder_branch(latent_input, start_shape, upsample_sizes, filters, branch_name):
    x = Dense(int(np.prod(start_shape)), activation='relu', name=f'{branch_name}_dec_dense')(latent_input)
    x = Reshape(start_shape, name=f'{branch_name}_dec_reshape')(x)

    for i, up_size in enumerate(upsample_sizes):
        x = UpSampling2D(size=up_size, name=f'{branch_name}_dec_up_{i+1}')(x)
        x = Conv2DTranspose(filters[i], kernel_size=(3, 3), padding='same', name=f'{branch_name}_dec_deconv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_dec_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_dec_relu_{i+1}')(x)

    x = Conv2D(1, kernel_size=(3, 3), padding='same', activation='linear', dtype='float32', name=f'{branch_name}_output')(x)
    return x


# ==========================================
# 5. MODELL ÖSSZEÁLLÍTÁSA (TEMPO NÉLKÜL)
# ==========================================
print("\n🚀 Autoencoder építése (Mel + Chroma)...")

input_mel    = Input(shape=(128, 1280, 1),  name='mel_input')
input_chroma = Input(shape=(12,  1280, 1),  name='chroma_input')

# Encoder (TÖRLÉS: branch_tempo)
branch_mel    = create_encoder_branch(input_mel, [32, 64, 128, 256], [(2, 4), (2, 4), (2, 4), (2, 2)], "mel")
branch_chroma = create_encoder_branch(input_chroma, [16, 32, 64], [(1, 4), (1, 4), (1, 4)], "chroma")

merged = Concatenate(name='concat_all_branches')([branch_mel, branch_chroma])

z = Dense(512, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)

# Bottleneck
bottleneck_raw = Dense(256, activation='relu', dtype='float32', name='bottleneck')(z)
bottleneck     = L2NormLayer(name='l2_norm')(bottleneck_raw)

# Decoder (TÖRLÉS: dec_tempo)
dec_mel    = create_decoder_branch(bottleneck, start_shape=(8, 10, 256), upsample_sizes=[(2, 2), (2, 4), (2, 4), (2, 4)], filters=[128, 64, 32, 16], branch_name="mel")
dec_chroma = create_decoder_branch(bottleneck, start_shape=(12, 20, 64), upsample_sizes=[(1, 4), (1, 4), (1, 4)], filters=[32, 16, 8], branch_name="chroma")

autoencoder = Model(
    inputs=[input_mel, input_chroma],
    outputs={'mel_output': dec_mel, 'chroma_output': dec_chroma},
    name='spotify_autoencoder_notempo'
)

# Újrasúlyozás a Tempo elvesztése miatt
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={'mel_output': 'mse', 'chroma_output': 'mse'},
    loss_weights={'mel_output': 1.0, 'chroma_output': 0.3},
    metrics={'mel_output': 'mae', 'chroma_output': 'mae'}
)

encoder = Model(
    inputs=[input_mel, input_chroma],
    outputs=bottleneck,
    name='encoder_only_notempo'
)

autoencoder.summary()


# ==========================================
# 6. TANÍTÁS
# ==========================================
print("\n⚙️ Generátorok inicializálása...")

BATCH_SIZE = 64 

train_gen = SpotifyDataGenerator(H5_PATH, split_name='train', batch_size=BATCH_SIZE)
val_gen   = SpotifyDataGenerator(H5_PATH, split_name='val',   batch_size=BATCH_SIZE, shuffle=False)

callbacks = [
    ModelCheckpoint(SAVE_PATH_AE, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-5)
]

print(f"\n🔥 GPU TANÍTÁS INDÍTÁSA... (Batch size: {BATCH_SIZE})")
history = autoencoder.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15, # Mivel nulláról indul, érdemes picit hosszabbra hagyni
    callbacks=callbacks
)

# ==========================================
# 7. ENCODER MENTÉSE
# ==========================================
encoder.save(SAVE_PATH_ENCODER)
print(f"\n✅ Encoder elmentve: {SAVE_PATH_ENCODER}")


🚀 Autoencoder építése (Mel + Chroma)...


Model: "spotify_autoencoder_notempo"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_input           │ (None, 128, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_conv_1      │ (None, 128, 1280, │        320 │ mel_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_bn_1        │ (None, 128, 1280, │        128 │ mel_enc_conv_1[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_relu_1      │ (None, 128, 1280, │          0 │ mel_enc_bn_1[0][… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_pool_1      │ (None, 64, 320,   │          0 │ mel_enc_relu_1[0… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_input        │ (None, 12, 1280,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_conv_2      │ (None, 64, 320,   │     18,496 │ mel_enc_pool_1[0… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_enc_conv_1   │ (None, 12, 1280,  │        160 │ chroma_input[0][… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_bn_2        │ (None, 64, 320,   │        256 │ mel_enc_conv_2[0… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_enc_bn_1     │ (None, 12, 1280,  │         64 │ chroma_enc_conv_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_relu_2      │ (None, 64, 320,   │          0 │ mel_enc_bn_2[0][… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_enc_relu_1   │ (None, 12, 1280,  │          0 │ chroma_enc_bn_1[… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_pool_2      │ (None, 32, 80,    │          0 │ mel_enc_relu_2[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_enc_pool_1   │ (None, 12, 320,   │          0 │ chroma_enc_relu_… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_conv_3      │ (None, 32, 80,    │     73,856 │ mel_enc_pool_2[0… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_enc_conv_2   │ (None, 12, 320,   │      4,640 │ chroma_enc_pool_… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_enc_bn_3        │ (None, 32, 80,    │        512 │ mel_enc_conv_3[0

 Total params: 10,503,522 (40.07 MB)

 Trainable params: 10,500,722 (40.06 MB)

 Non-trainable params: 2,800 (10.94 KB)


⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

🔥 GPU TANÍTÁS INDÍTÁSA... (Batch size: 64)
Epoch 1/15


I0000 00:00:1773908187.845925     120 service.cc:152] XLA service 0x7af2d0001f20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773908187.845970     120 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773908190.379704     120 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773908244.514653     120 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - chroma_output_loss: 0.1162 - chroma_output_mae: 0.2560 - loss: 0.0828 - mel_output_loss: 0.0478 - mel_output_mae: 0.1361
Epoch 1: val_loss improved from inf to 0.06110, saving model to /kaggle/working/spotify_autoencoder_notempo.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 815s 2s/step - chroma_output_loss: 0.1161 - chroma_output_mae: 0.2559 - loss: 0.0827 - mel_output_loss: 0.0477 - mel_output_mae: 0.1360 - val_chroma_output_loss: 0.1081 - val_chroma_output_mae: 0.2473 - val_loss: 0.0611 - val_mel_output_loss: 0.0287 - val_mel_output_mae: 0.1360 - learning_rate: 0.0010
Epoch 2/15
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - chroma_output_loss: 0.0665 - chroma_output_mae: 0.2069 - loss: 0.0327 - mel_output_loss: 0.0128 - mel_output_mae: 0.0880
Epoch 2: val_loss improved from 0.06110 to 0.03336, saving model to /kaggle/working/spotify_autoencoder_notempo.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 598s 2s/step - chroma_output_loss: 0.0665 - chroma_output_mae: 0.2069 

## Kiértékelés

In [1]:
import os
import h5py
import numpy as np
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
import random

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model" 
AE_VECTORS_PATH = "../Models/ae_vectors_closed_world_notempo.npy" # <-- ÚJ: A kimentett vektorok helye

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500]
NUM_TEST_SAMPLES = 1000
STEP_SIZE = 2

# ==========================================
# 1. SEGÉDFÜGGVÉNYEK
# ==========================================
def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def calculate_auc_fast(sims_array, relevant_indices, valid_indices, num_negatives=1000):
    pos_scores = [sims_array[idx] for idx in relevant_indices]
    if not pos_scores: return 0.5
        
    possible_negatives = list(set(valid_indices) - set(relevant_indices))
    n_neg = min(num_negatives, len(possible_negatives))
    
    neg_scores = []
    if n_neg > 0:
        chosen_negatives = random.sample(possible_negatives, n_neg)
        neg_scores = [sims_array[idx] for idx in chosen_negatives]
    
    if not neg_scores: return 0.5
        
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 2. FŐPROGRAM
# ==========================================
def main():
    print("1. Word2Vec és AE vektorok betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    
    # --- ÚJ: Autoencoder helyett csak a mátrixot töltjük be ---
    ae_vectors = np.load(AE_VECTORS_PATH)
    print(f" -> Autoencoder vektorok betöltve. Alakja: {ae_vectors.shape}")

    print("\n2. Zárt Világ (Closed World) halmaz meghatározása...")
    with h5py.File(H5_PATH, "r") as hf:
        all_uris_bytes = hf["ml/track_uri"][:]
        all_uris = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in all_uris_bytes])
        splits = hf["ml/split"][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])

    w2v_vocab = set(w2v_model.wv.key_to_index.keys())
    valid_uris = set(all_uris).intersection(w2v_vocab)
    
    uri_to_index = {uri: i for i, uri in enumerate(all_uris)}
    valid_indices = [uri_to_index[uri] for uri in valid_uris]
    
    print(f" -> Érvényes dalok a Zárt Világban: {len(valid_uris):,} db")

    print("\n3. Nyers (Raw) Akusztikus adatbázis legenerálása...")
    # Az AE vektorok már megvannak, így itt csak a baseline "Raw" vektorokat számoljuk ki.
    raw_vectors_list = []
    
    with h5py.File(H5_PATH, "r") as hf:
        batch_size = 128 # Nagyobb batch size is lehet, mivel csak átlagot vonunk
        for i in tqdm(range(0, len(all_uris), batch_size), desc="Nyers jellemzők kinyerése"):
            mel_batch = hf["spectrograms/mel"][i:i+batch_size]
            chroma_batch = hf["spectrograms/chroma"][i:i+batch_size]
            
            mel_raw = np.mean(mel_batch, axis=1)
            chroma_raw = np.mean(chroma_batch, axis=1)
            
            raw_concat = np.concatenate([mel_raw, chroma_raw], axis=1)
            raw_norms = np.linalg.norm(raw_concat, axis=1, keepdims=True)
            raw_concat_norm = np.divide(raw_concat, raw_norms, out=np.zeros_like(raw_concat), where=raw_norms!=0)
            
            raw_vectors_list.extend(raw_concat_norm)

    raw_vectors = np.array(raw_vectors_list, dtype=np.float32)

    w2v_matrix = np.zeros((len(all_uris), w2v_model.vector_size), dtype=np.float32)
    for i, uri in enumerate(all_uris):
        if uri in w2v_model.wv:
            w2v_matrix[i] = w2v_model.wv[uri]

    print("\n4. Ground Truth szótárak építése...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)

    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        gt_uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            u = gt_uris[i].decode('utf-8') if isinstance(gt_uris[i], bytes) else gt_uris[i]
            track_to_playlists[u].add(pid)
            playlist_to_tracks[pid].add(u)

    print("\n5. Teszt halmaz kinyerése...")
    test_indices = [i for i, s in enumerate(splits_str) if s == "test" and all_uris[i] in valid_uris]
    test_indices = test_indices[::STEP_SIZE]
    
    if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
        test_indices = test_indices[:NUM_TEST_SAMPLES]

    results = {
        "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "raw":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "ae":       {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
    }

    print("\n6. Kiértékelés futtatása (Fair, Closed-World mód)...")
    for idx in tqdm(test_indices, desc="Dalok tesztelése"):
        query_uri = all_uris[idx]
        
        pids = track_to_playlists.get(query_uri, set())
        if not pids: continue
            
        relevant_uris = set()
        for pid in pids:
            relevant_uris.update(playlist_to_tracks[pid])
        relevant_uris.discard(query_uri)
        relevant_uris = relevant_uris.intersection(valid_uris)
        
        if not relevant_uris: continue
        
        relevant_indices = [uri_to_index[u] for u in relevant_uris]

        # --- A) BASELINE (Word2Vec) ---
        query_w2v = w2v_matrix[idx].reshape(1, -1)
        sims_w2v = cosine_similarity(query_w2v, w2v_matrix)[0]
        sorted_indices_w2v = np.argsort(sims_w2v)[::-1]
        
        w2v_recs = [all_uris[i] for i in sorted_indices_w2v if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["baseline"]["auc"].append(calculate_auc_fast(sims_w2v, relevant_indices, valid_indices))

        # --- B) NYERS AUDIÓ BASELINE (RAW) ---
        query_raw = raw_vectors[idx].reshape(1, -1)
        sims_raw = cosine_similarity(query_raw, raw_vectors)[0]
        sorted_indices_raw = np.argsort(sims_raw)[::-1]
        
        raw_recs = [all_uris[i] for i in sorted_indices_raw if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["raw"]["auc"].append(calculate_auc_fast(sims_raw, relevant_indices, valid_indices))

        # --- C) 3-ÁGÚ AUTOENCODER KERESŐMOTOR ---
        # Itt már a betöltött numpy array-t használjuk közvetlenül!
        query_ae = ae_vectors[idx].reshape(1, -1)
        sims_ae = cosine_similarity(query_ae, ae_vectors)[0]
        sorted_indices_ae = np.argsort(sims_ae)[::-1]
        
        ae_recs = [all_uris[i] for i in sorted_indices_ae if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["ae"]["auc"].append(calculate_auc_fast(sims_ae, relevant_indices, valid_indices))

        # --- METRIKÁK KISZÁMÍTÁSA ---
        for k in K_VALUES:
            hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
            results["baseline"][k]["hr"].append(hr_b)
            results["baseline"][k]["prec"].append(prec_b)
            results["baseline"][k]["rec"].append(rec_b)
            
            hr_r, prec_r, rec_r = calculate_metrics(raw_recs, relevant_uris, k)
            results["raw"][k]["hr"].append(hr_r)
            results["raw"][k]["prec"].append(prec_r)
            results["raw"][k]["rec"].append(rec_r)

            hr_ae, prec_ae, rec_ae = calculate_metrics(ae_recs, relevant_uris, k)
            results["ae"][k]["hr"].append(hr_ae)
            results["ae"][k]["prec"].append(prec_ae)
            results["ae"][k]["rec"].append(rec_ae)

    # --- 7. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*75)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆")
    print("="*75)
    
    print(f"\n✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):")
    print(f"  1. BASELINE (Word2Vec): {np.mean(results['baseline']['auc'])*100:05.2f}%")
    print(f"  2. NYERS AUDIÓ (Raw):   {np.mean(results['raw']['auc'])*100:05.2f}%")
    print(f"  3. AUTOENCODER (AE):    {np.mean(results['ae']['auc'])*100:05.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("1. BASELINE (Word2Vec / Lejátszási Listák):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:05.2f}%")
        
        print("2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):")
        print(f"  Hit Rate@{k}:  {np.mean(results['raw'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['raw'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['raw'][k]['rec'])*100:05.2f}%")

        print("3. AUTOENCODER (Mélytanulásos Akusztika):")
        print(f"  Hit Rate@{k}:  {np.mean(results['ae'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['ae'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['ae'][k]['rec'])*100:05.2f}%")
    print("\n" + "="*75)

if __name__ == "__main__":
    main()

1. Word2Vec és AE vektorok betöltése...
 -> Autoencoder vektorok betöltve. Alakja: (27052, 256)

2. Zárt Világ (Closed World) halmaz meghatározása...
 -> Érvényes dalok a Zárt Világban: 25,511 db

3. Nyers (Raw) Akusztikus adatbázis legenerálása...


Nyers jellemzők kinyerése: 100%|██████████| 212/212 [00:40<00:00,  5.21it/s]



4. Ground Truth szótárak építése...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:25<00:00, 456180.91it/s]



5. Teszt halmaz kinyerése...

6. Kiértékelés futtatása (Fair, Closed-World mód)...


Dalok tesztelése: 100%|██████████| 1000/1000 [08:04<00:00,  2.07it/s]


🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆

✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):
  1. BASELINE (Word2Vec): 76.80%
  2. NYERS AUDIÓ (Raw):   49.27%
  3. AUTOENCODER (AE):    55.03%

--- TOP 1 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@1:  87.10% | Prec: 87.10% | Rec: 00.05%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@1:  20.40% | Prec: 20.40% | Rec: 00.00%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@1:  37.60% | Prec: 37.60% | Rec: 00.02%

--- TOP 5 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@5:  96.30% | Prec: 79.28% | Rec: 00.22%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@5:  50.60% | Prec: 17.92% | Rec: 00.02%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@5:  72.40% | Prec: 32.82% | Rec: 00.05%

--- TOP 10 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@10:  97.70% | Prec: 76.19% | Rec: 00.41%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rat